In [4]:
!pip install craft-text-detector

  Using cached craft_text_detector-0.4.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached torchvision-0.24.0-cp312-cp312-win_amd64.whl.metadata (5.9 kB)
  Using cached opencv-python-4.5.4.60.tar.gz (89.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'error'


  error: subprocess-exited-with-error
  
  × installing build dependencies for opencv-python did not run successfully.
  │ exit code: 1
  ╰─> [19 lines of output]
      Ignoring numpy: markers 'python_version == "3.6" and platform_machine != "aarch64" and platform_machine != "arm64"' don't match your environment
      Ignoring numpy: markers 'python_version == "3.7" and platform_machine != "aarch64" and platform_machine != "arm64"' don't match your environment
      Ignoring numpy: markers 'python_version == "3.8" and platform_machine != "aarch64" and platform_machine != "arm64"' don't match your environment
      Ignoring numpy: markers 'python_version <= "3.9" and sys_platform == "linux" and platform_machine == "aarch64"' don't match your environment
      Ignoring numpy: markers 'python_version <= "3.9" and sys_platform == "darwin" and platform_machine == "arm64"' don't match your environment
      Ignoring numpy: markers 'python_version == "3.9" and platform_machine != "aarch64" an

In [7]:
#replacing pytessaract with CRAFT for OCR
import re
import cv2
import torch
import numpy as np
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from craft_text_detector import Craft

# -----------------------------
# Load DistilBERT Model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = "./results"  # path to your trained DistilBERT model
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()

# -----------------------------
# Regex patterns
# -----------------------------
regex_patterns = {
    "aws_access": r'\bAKIA[0-9A-Z]{16}\b',
    "aws_secret": r'\b[A-Za-z0-9/+=]{40}\b',
    "email": r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
    "ssn": r'\b\d{3}-\d{2}-\d{4}\b',
    "credit_card": r'\b(?:\d[ -]*?){13,16}\b',
    "phone": r'\+?\d[\d\s-]{7,}\d',
    "api_key": r'\b[A-Za-z0-9]{20,}\b'
}

def is_sensitive_by_regex(text):
    for label, pattern in regex_patterns.items():
        if re.search(pattern, text):
            return True
    return False

def is_sensitive_by_bert(text):
    if not text.strip():
        return False
    enc = tokenizer(text, truncation=True, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**enc)
        pred = torch.argmax(outputs.logits, dim=-1).item()
    return bool(pred)  # 1 = sensitive, 0 = non-sensitive


# -----------------------------
# Redaction using CRAFT
# -----------------------------
def redact_with_craft(image_path, output_path="redacted_output.png", blur_strength=(51, 51)):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")

    craft = Craft(output_dir=None, crop_type="poly", cuda=torch.cuda.is_available())
    prediction_result = craft.detect_text(image_path)

    boxes = prediction_result["boxes"]
    texts = prediction_result["text"]

    sensitive_boxes = []

    for box, text in zip(boxes, texts):
        if not text.strip():
            continue
        regex_flag = is_sensitive_by_regex(text)
        bert_flag = is_sensitive_by_bert(text)

        if regex_flag and bert_flag:
            sensitive_boxes.append(box)
            print(f"🔒 Sensitive detected: '{text}'")

    # Apply Gaussian blur on detected sensitive boxes
    for box in sensitive_boxes:
        x_min = int(np.min(box[:, 0]))
        y_min = int(np.min(box[:, 1]))
        x_max = int(np.max(box[:, 0]))
        y_max = int(np.max(box[:, 1]))

        roi = img[y_min:y_max, x_min:x_max]
        blurred_roi = cv2.GaussianBlur(roi, blur_strength, 30)
        img[y_min:y_max, x_min:x_max] = blurred_roi

    cv2.imwrite(output_path, img)
    print(f"✅ Redacted image saved as {output_path}")

    craft.unload_craftnet_model()
    craft.unload_refinenet_model()


# -----------------------------
# Run Pipeline
# -----------------------------
if __name__ == "__main__":
    image_path = "test1.png"
    redact_with_craft(image_path, "redacted_output.png")

ModuleNotFoundError: No module named 'craft_text_detector'